# Self Supervised and Contrastive Representation Learning on STL10

## Part A: Supervised Learning with Limited Labels




In [1]:
# Step 0: Personal parameters

SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)


SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3


## Environment and Reproducibility

In [2]:
import os
import random
import json
import time
import platform
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Reproducibility seed:", SEED)


Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
Reproducibility seed: 2046


## Load STL10




In [3]:
DATA_ROOT = "./data"

STL10_MEAN = (0.4467, 0.4398, 0.4066)
STL10_STD = (0.2603, 0.2566, 0.2713)

train_transform = transforms.Compose([
    transforms.RandomCrop(
        96,
        padding=12,
        padding_mode="reflect",
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        STL10_MEAN,
        STL10_STD,
    ),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        STL10_MEAN,
        STL10_STD,
    ),
])

train_full = datasets.STL10(
    root=DATA_ROOT,
    split="train",
    download=True,
    transform=train_transform,
)

test_dataset = datasets.STL10(
    root=DATA_ROOT,
    split="test",
    download=True,
    transform=test_transform,
)

print("Full labeled train size:", len(train_full))
print("Test size:", len(test_dataset))

assert len(train_full) == 5000
assert len(test_dataset) == 8000


100%|██████████| 2.64G/2.64G [08:02<00:00, 5.47MB/s]


Full labeled train size: 5000
Test size: 8000


## Select the Required 500 Labeled Images

In [4]:
LABELED_SUBSET_SIZE = 500

subset_rng = np.random.default_rng(SEED)

labeled_indices = subset_rng.choice(
    len(train_full),
    size=LABELED_SUBSET_SIZE,
    replace=False,
)

labeled_indices = labeled_indices.tolist()

train_subset = Subset(
    train_full,
    labeled_indices,
)

os.makedirs("artifacts", exist_ok=True)

with open(
    "artifacts/labeled_500_indices.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(labeled_indices, f, indent=2)

print("Selected labeled images:", len(train_subset))
print("First 10 selected indices:", labeled_indices[:10])

assert len(train_subset) == 500
assert len(set(labeled_indices)) == 500


Selected labeled images: 500
First 10 selected indices: [633, 1659, 3085, 3851, 2592, 2255, 263, 4290, 1272, 1065]


## Inspect the Limited Label Subset



In [5]:
selected_targets = np.array(
    train_full.labels
)[labeled_indices]

class_counts = pd.Series(
    selected_targets
).value_counts().sort_index()

class_distribution = pd.DataFrame({
    "Class ID": list(range(10)),
    "Class Name": train_full.classes,
    "Count": [
        int(class_counts.get(i, 0))
        for i in range(10)
    ],
})

class_distribution


,Class ID,Class Name,Count
0,0,airplane,47
1,1,bird,54
2,2,car,50
3,3,cat,43
4,4,deer,51
5,5,dog,48
6,6,horse,43
7,7,monkey,62
8,8,ship,51
9,9,truck,51


## Data Loaders

In [6]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    generator=train_generator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))

print("Training batch image shape:", images.shape)
print("Training batch label shape:", labels.shape)
print("Label range:", int(labels.min()), "to", int(labels.max()))

assert images.shape[1:] == (3, 96, 96)
assert labels.min() >= 0
assert labels.max() <= 9


Training batch image shape: torch.Size([64, 3, 96, 96])
Training batch label shape: torch.Size([64])
Label range: 0 to 9


## Part A Model: ResNet18 from Scratch




In [7]:
supervised_model = resnet18(
    weights=None
)

supervised_model.fc = nn.Linear(
    supervised_model.fc.in_features,
    10,
)

supervised_model = supervised_model.to(device)

total_params = sum(
    p.numel()
    for p in supervised_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in supervised_model.parameters()
    if p.requires_grad
)

print("Architecture: ResNet18")
print("Pretrained weights: None")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

assert total_params == trainable_params


Architecture: ResNet18
Pretrained weights: None
Total parameters: 11,181,642
Trainable parameters: 11,181,642


## Training Setup

In [8]:
NUM_EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    supervised_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
)

print("Epochs:", NUM_EPOCHS)
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Loss: CrossEntropyLoss")


Epochs: 12
Optimizer: Adam
Learning rate: 0.001
Weight decay: 0.0001
Loss: CrossEntropyLoss


## Train the Supervised Baseline

In [9]:
def evaluate_accuracy(model, loader):
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            logits = model(images)
            loss = criterion(logits, labels)

            total_loss += (
                float(loss.detach().cpu())
                * labels.size(0)
            )

            predictions = logits.argmax(dim=1)

            correct += int(
                (predictions == labels).sum().item()
            )
            total += labels.size(0)

    return {
        "loss": total_loss / total,
        "accuracy": correct / total,
    }


In [10]:
training_history = []
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    supervised_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_index, (images, labels) in enumerate(
        train_loader,
        start=1,
    ):
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = supervised_model(images)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite loss at epoch "
                f"{epoch}, batch {batch_index}"
            )

        loss.backward()
        optimizer.step()

        running_loss += (
            float(loss.detach().cpu())
            * labels.size(0)
        )

        predictions = logits.argmax(dim=1)

        correct += int(
            (predictions == labels).sum().item()
        )
        total += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    training_history.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "train_accuracy": epoch_accuracy,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"loss={epoch_loss:.4f} | "
        f"train_acc={epoch_accuracy * 100:.2f}% | "
        f"lr={scheduler.get_last_lr()[0]:.6f}"
    )

training_time = time.time() - training_start

print()
print(
    f"Training time: "
    f"{training_time:.2f} seconds"
)


Epoch 01/12 | loss=2.2202 | train_acc=21.20% | lr=0.000983
Epoch 02/12 | loss=1.9304 | train_acc=28.40% | lr=0.000933
Epoch 03/12 | loss=1.7481 | train_acc=31.60% | lr=0.000854
Epoch 04/12 | loss=1.4766 | train_acc=44.20% | lr=0.000750
Epoch 05/12 | loss=1.3791 | train_acc=47.80% | lr=0.000629
Epoch 06/12 | loss=1.2751 | train_acc=53.00% | lr=0.000500
Epoch 07/12 | loss=1.1599 | train_acc=59.60% | lr=0.000371
Epoch 08/12 | loss=1.0287 | train_acc=62.80% | lr=0.000250
Epoch 09/12 | loss=0.9711 | train_acc=64.40% | lr=0.000146
Epoch 10/12 | loss=0.8425 | train_acc=72.60% | lr=0.000067
Epoch 11/12 | loss=0.8401 | train_acc=70.80% | lr=0.000017
Epoch 12/12 | loss=0.7395 | train_acc=74.60% | lr=0.000000

Training time: 10.83 seconds


## Training History

In [11]:
history_df = pd.DataFrame(
    training_history
)

history_df


,epoch,train_loss,train_accuracy,learning_rate
0,1,2.220212,0.212,0.000983
1,2,1.930433,0.284,0.000933
2,3,1.748150,0.316,0.000854
3,4,1.476578,0.442,0.000750
4,5,1.379070,0.478,0.000629
5,6,1.275072,0.530,0.000500
6,7,1.159878,0.596,0.000371
7,8,1.028689,0.628,0.000250
8,9,0.971077,0.644,0.000146
9,10,0.842466,0.726,0.000067


## Part A Test Accuracy

In [12]:
test_metrics = evaluate_accuracy(
    supervised_model,
    test_loader,
)

test_accuracy = test_metrics["accuracy"]
test_loss = test_metrics["loss"]

print(
    f"Test loss: "
    f"{test_loss:.4f}"
)
print(
    f"Test accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

assert np.isfinite(test_loss)
assert 0.0 <= test_accuracy <= 1.0


Test loss: 1.5381
Test accuracy: 45.31%


## Part A Interpretation

The supervised model was trained end to end from random initialization using only 500 labeled STL10 images. The measured test accuracy above is the supervised limited label baseline that the Rotation SSL and SimCLR representations will be compared against later.


## Save Checkpoint and Part A Artifacts

In [13]:
os.makedirs(
    "checkpoints",
    exist_ok=True,
)

checkpoint_path = (
    "checkpoints/supervised_resnet18.pt"
)

torch.save(
    {
        "model_state_dict": (
            supervised_model.state_dict()
        ),
        "architecture": "resnet18",
        "num_classes": 10,
        "pretrained": False,
        "SID4": SID4,
        "SEED": SEED,
        "labeled_indices": labeled_indices,
        "epochs": NUM_EPOCHS,
        "test_accuracy": test_accuracy,
    },
    checkpoint_path,
)

history_df.to_csv(
    "artifacts/part_a_training_history.csv",
    index=False,
)

part_a_metrics = {
    "timestamp": datetime.now().isoformat(),
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
    "dataset": "STL10",
    "architecture": "ResNet18",
    "pretrained_weights": False,
    "labeled_training_images": 500,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "training_time_seconds": training_time,
}

with open(
    "artifacts/part_a_metrics.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(
        part_a_metrics,
        f,
        indent=2,
    )

print(
    json.dumps(
        part_a_metrics,
        indent=2,
    )
)

print()
print(
    "Checkpoint saved:",
    checkpoint_path,
)


{
  "timestamp": "2026-09-30T19:19:57.915195",
  "SID4": 2046,
  "SEED": 2046,
  "SLICE": 46,
  "HP_ID": 0,
  "CLS_A": 6,
  "CLS_B": 3,
  "dataset": "STL10",
  "architecture": "ResNet18",
  "pretrained_weights": false,
  "labeled_training_images": 500,
  "epochs": 12,
  "batch_size": 64,
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "test_loss": 1.538141303062439,
  "test_accuracy": 0.453125,
  "training_time_seconds": 10.826482772827148
}

Checkpoint saved: checkpoints/supervised_resnet18.pt
